# Part A evidence log — review of the inherited route classifier

The previous engineer's `baseline/baseline_classifier.py` reports **98.75% accuracy** and
recommends shipping. This notebook is the evidence log behind my review: every claim I make in
the README is *measured* here, on the real data, before I assume it. Structure:

1. The dataset at a glance
2. Templates and duplicates — why a random split flatters
3. Reproducing 98.75% and testing the leakage claim
4. What an 80-row test set can certify
5. The honest protocol: template-grouped CV, per-class metrics
6. Capacity: 1,522 features for 400 rows
7. Adversarial probe: generating false positives from the learned features
8. Regularization as an operating point
9. Calibration and the human-review band
10. Alternatives (incl. Naive Bayes), with significance testing
11. Hygiene candidate: `min_df=2`
12. Embeddings — considered, rejected
13. Verdict

In [1]:
import csv, re, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, recall_score)

SEED = 0
DATA = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
rows = list(csv.DictReader(open(DATA, encoding="utf-8")))
texts = np.array([r["text"] for r in rows], dtype=object)
labels = np.array([r["label"] for r in rows])
ROUTES = sorted(set(labels))

def baseline_config(**kw):
    """The previous engineer's exact model config, as a leakage-proof pipeline."""
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=kw.get("min_df", 1), sublinear_tf=True),
        LogisticRegression(max_iter=2000, C=10.0, class_weight=kw.get("class_weight")),
    )

import sklearn
print(f"{len(rows)} rows | sklearn {sklearn.__version__} | numpy {np.__version__}")

400 rows | sklearn 1.8.0 | numpy 1.26.4


## 1. The dataset at a glance

Four routes with a 3.2:1 spread between the most and least common class. `fraud-report` — the
route the brief says is *most expensive to get wrong* — is the rarest at 12.5%. Any evaluation
that blends the classes (accuracy) is structurally biased against noticing fraud mistakes.

In [2]:
counts = pd.Series(labels).value_counts()
summary = pd.DataFrame({
    "count": counts,
    "share": (counts / len(labels)).round(3),
    "balanced_class_weight n/(K*n_c)": (len(labels) / (len(ROUTES) * counts)).round(3),
})
lengths = pd.Series([len(t.split()) for t in texts])
print(summary, "\n")
print(f"message length (words): min {lengths.min()}, median {lengths.median():.0f}, max {lengths.max()}")

                     count  share  balanced_class_weight n/(K*n_c)
general                160  0.400                            0.625
account-access         100  0.250                            1.000
transaction-dispute     90  0.225                            1.111
fraud-report            50  0.125                            2.000 

message length (words): min 8, median 16, max 24


## 2. Templates and duplicates — why a random split flatters

Reading the raw file shows generator fingerprints: the same sentence with only the asset name
swapped ("How does staking work ... on Polygon / Ethereum / Solana"). If sibling copies of one
template land on both sides of a random split, the "held-out" set partially *is* the training
set. I normalize the text, mask asset names, and count.

The `groups` array built here drives every honest evaluation below: **GroupKFold keeps all
members of a template on the same side**, so the model is always scored on *phrasings it never
saw*.

In [3]:
def norm(t):
    t = re.sub(r"[^a-z ]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()

ASSET = (r"\b(btc|eth|sol|ada|doge|dogecoin|bitcoin|ethereum|solana|polygon|matic|xrp|usdc|"
         r"ltc|litecoin|avax|cardano|tron|dot|polkadot)\b")
normed = [norm(t) for t in texts]
tmpl = [re.sub(ASSET, "ASSET", s) for s in normed]

from collections import Counter
exact = Counter(normed); templ = Counter(tmpl)
print(f"exact duplicate rows (normalized): {sum(v for v in exact.values() if v > 1)} "
      f"in {sum(1 for v in exact.values() if v > 1)} groups")
print(f"template duplicate rows (asset-masked): {sum(v for v in templ.values() if v > 1)} "
      f"in {sum(1 for v in templ.values() if v > 1)} groups\n")

example = max(templ, key=templ.get)  # largest template group
print("example template group:")
for t in texts:
    if re.sub(ASSET, "ASSET", norm(t)) == example:
        print("  -", t)

gid = {}
groups = np.array([gid.setdefault(s, len(gid)) for s in tmpl])
print(f"\n{len(set(groups))} template groups for {len(texts)} rows")

exact duplicate rows (normalized): 4 in 2 groups
template duplicate rows (asset-masked): 28 in 14 groups

example template group:
  - Hi, How long do SOL withdrawals usually take to process?
  - Hi, How long do Bitcoin withdrawals usually take to process?

386 template groups for 400 rows


## 3. Reproducing 98.75% and testing the leakage claim

The shipped script calls `fit_transform` on **all 400 rows and splits afterwards**, so the
TF-IDF vocabulary and IDF weights were computed with the test rows included. That is
preprocessing leakage — the eval saw the test set. Rule: *a valid evaluation must simulate the
information conditions of deployment*; `Pipeline` enforces it.

But I verify the *size* of the effect instead of assuming it.

In [4]:
# (a) exactly as shipped: fit on everything, then split
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X_leaky = vec.fit_transform(texts)
Xtr, Xte, ytr, yte = train_test_split(X_leaky, labels, test_size=0.2, random_state=SEED)
leaky = accuracy_score(yte, LogisticRegression(max_iter=2000, C=10.0).fit(Xtr, ytr).predict(Xte))

# (b) same seed, split FIRST, vectorizer fitted on train only
ttr, tte, y2tr, y2te = train_test_split(texts, labels, test_size=0.2, random_state=SEED)
clean = accuracy_score(y2te, baseline_config().fit(ttr, y2tr).predict(tte))

# (c) how different are the learned statistics?
v_all = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(texts)
v_tr  = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(list(ttr))
only_in_full = set(v_all.get_feature_names_out()) - set(v_tr.get_feature_names_out())
idf_all = dict(zip(v_all.get_feature_names_out(), v_all.idf_))
idf_tr  = dict(zip(v_tr.get_feature_names_out(),  v_tr.idf_))

print(f"as shipped (leaky protocol):        {leaky:.4f}")
print(f"split-first, same seed (no leak):   {clean:.4f}")
print(f"vocabulary entries that exist ONLY because the fit saw test rows: {len(only_in_full)}")
print("\nidf drift for sample terms (all-400 fit vs train-only fit):")
for t in ["login", "refund", "fraud"]:
    print(f"  {t:<8} {idf_all[t]:.3f}  vs  {idf_tr[t]:.3f}")

as shipped (leaky protocol):        0.9875
split-first, same seed (no leak):   0.9875
vocabulary entries that exist ONLY because the fit saw test rows: 96

idf drift for sample terms (all-400 fit vs train-only fit):
  login    3.356  vs  3.337
  refund   3.698  vs  3.727
  fraud    4.915  vs  4.980


**Reading:** the protocol is invalid, but removing the leak does not move the score — the IDF
drift is in the third decimal. The honest statement is *"this is a protocol violation whose
effect happens to be negligible on this dataset"* — not "the real number is much lower". The
next sections show why the number survives (the task is at ceiling) and why it still cannot be
trusted as a ship signal.

## 4. What can an 80-row test set certify?

One split, one seed, n=80. Two ways to see the same problem: the spread across seeds, and the
binomial (Wilson) confidence interval around 79/80.

In [5]:
accs = []
for s in range(20):
    a, b, c, d = train_test_split(texts, labels, test_size=0.2, random_state=s)
    accs.append(accuracy_score(d, baseline_config().fit(a, c).predict(b)))
print(f"split-first accuracy across 20 seeds: min {min(accs):.4f}  mean {np.mean(accs):.4f}  max {max(accs):.4f}")

from statsmodels.stats.proportion import proportion_confint
lo, hi = proportion_confint(79, 80, method="wilson")
print(f"Wilson 95% CI for the reported 79/80: [{lo:.3f}, {hi:.3f}]  (width {100*(hi-lo):.1f} pp)")
print("=> '98.75%' is statistically indistinguishable from 94%. One message = 1.25 pp.")

split-first accuracy across 20 seeds: min 0.9625  mean 0.9925  max 1.0000
Wilson 95% CI for the reported 79/80: [0.933, 0.998]  (width 6.5 pp)
=> '98.75%' is statistically indistinguishable from 94%. One message = 1.25 pp.


## 5. The honest protocol: template-grouped CV, per-class metrics

Grouped 5-fold CV (no template straddles a fold), the previous engineer's exact model config,
and the full per-class picture their report never showed.

In [6]:
gkf = GroupKFold(n_splits=5)
pred_grouped = cross_val_predict(baseline_config(), texts, labels, cv=gkf, groups=groups)
print(classification_report(labels, pred_grouped, digits=3))
print(pd.DataFrame(confusion_matrix(labels, pred_grouped, labels=ROUTES), index=ROUTES, columns=ROUTES))
print("\nthe single misrouted message:")
for t, y, p in zip(texts, labels, pred_grouped):
    if y != p:
        print(f"  true={y}  pred={p}\n  {t}")

                     precision    recall  f1-score   support

     account-access      1.000     1.000     1.000       100
       fraud-report      1.000     0.980     0.990        50
            general      1.000     1.000     1.000       160
transaction-dispute      0.989     1.000     0.994        90

           accuracy                          0.998       400
          macro avg      0.997     0.995     0.996       400
       weighted avg      0.998     0.998     0.997       400

                     account-access  fraud-report  general  transaction-dispute
account-access                  100             0        0                    0
fraud-report                      0            49        0                    1
general                           0             0      160                    0
transaction-dispute               0             0        0                   90

the single misrouted message:
  true=fraud-report  pred=transaction-dispute
  Hi, My account shows activity 

**Reading:** 399/400 under the honest protocol — the task is essentially linearly separable, so
the leakage never had anything to inflate. But look at *which* message fails: an urgent
**$10,000 theft routed to the dispute queue** — precisely the most expensive direction. The
per-class view, not the aggregate, is where the ship decision lives. (With
`class_weight='balanced'` this one error also disappears — Section 10 — but at n=50 fraud rows,
recall is only known to about ±7 pp either way.)

## 6. Capacity: 1,522 features for 400 rows

`min_df=1` keeps every term that appears even once. Features that occur in exactly **one**
training document are pure memorization hooks: maximally rare, maximally high-IDF, attachable
to whatever label that one row has. With weak regularization (C=10) the model is free to use
them, and with 3.8 features per row it can draw a perfect boundary — 100% *training* accuracy.

The per-class coefficients show what was actually learned.

In [7]:
vec_full = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(texts)
Xf = vec_full.transform(texts)
df_counts = np.asarray((Xf > 0).sum(axis=0)).ravel()
clf_full = LogisticRegression(max_iter=2000, C=10.0).fit(Xf, labels)
feats = np.array(vec_full.get_feature_names_out())

print(f"features: {Xf.shape[1]}  |  rows: {Xf.shape[0]}  |  ratio {Xf.shape[1]/Xf.shape[0]:.1f}:1")
print(f"features appearing in exactly ONE document: {(df_counts == 1).sum()}")
print(f"training accuracy (fit on all 400): {clf_full.score(Xf, labels):.4f}\n")

top = {cls: ", ".join(feats[np.argsort(clf_full.coef_[i])[::-1][:8]])
       for i, cls in enumerate(clf_full.classes_)}
print(pd.Series(top).to_string())

features: 1522  |  rows: 400  |  ratio 3.8:1
features appearing in exactly ONE document: 387
training accuracy (fit on all 400): 1.0000

account-access         login, reset, password, access, code, working,...
fraud-report           someone, account and, was, my account, fraud, ...
general                do, how, what, can, what the, how do, document...
transaction-dispute    but, refund, want, than, shows, but the, the, ...


**Reading:** apart from `fraud` and `refund`, the strongest votes are function words and
template fragments — `someone`, `was`, `never` for fraud; `but`, `shows`, `the` for disputes;
`how do` for general. The model has learned **the fingerprints of a synthetic data generator**,
not the concept of fraud. That is fine for this benchmark and exactly why the score must not be
read as production readiness.

## 7. Adversarial probe: the learned features generate false positives

If Section 6 is right, I should be able to write *benign* messages that trip the classifier by
stuffing them with fraud-vote tokens (`someone`, `my account and`, `never`, `was`) while keeping
the meaning harmless. No search, no gradient tricks — hand-written first attempts:

In [8]:
pipe_full = baseline_config().fit(texts, labels)
probes = [
    "I was wondering if someone on my account and I can both use the app? I never tried before.",
    "My account was set up by someone at your kiosk and I think I never got the welcome email.",
    "Someone told me staking was good. I think my account and wallet never earned anything though?",
    "I think the fraud protection features are great, someone recommended them to me.",
    "Was there ever a time someone could open an account and never verify their email?",
    "I want a refund policy explanation but the app shows nothing than a spinner.",
]
for p in probes:
    pred = pipe_full.predict([p])[0]
    conf = pipe_full.predict_proba([p]).max()
    print(f"pred={pred:<20} conf={conf:.3f} | {p}")

pred=fraud-report         conf=0.574 | I was wondering if someone on my account and I can both use the app? I never tried before.
pred=fraud-report         conf=0.756 | My account was set up by someone at your kiosk and I think I never got the welcome email.
pred=fraud-report         conf=0.948 | Someone told me staking was good. I think my account and wallet never earned anything though?
pred=fraud-report         conf=0.843 | I think the fraud protection features are great, someone recommended them to me.
pred=fraud-report         conf=0.786 | Was there ever a time someone could open an account and never verify their email?
pred=transaction-dispute  conf=0.963 | I want a refund policy explanation but the app shows nothing than a spinner.


**Reading:** five harmless messages classify as `fraud-report` — one at **0.948 confidence**
for what is plainly a staking question — and the `but/shows/refund` probe flips to
`transaction-dispute` at 0.963. First attempts, no optimization. A model this easy to steer
with function words will misfire on real traffic (and real fraud language is *adversarial*).
This is the concrete demonstration behind "the benchmark score does not transfer".

## 8. Regularization is an operating point

sklearn's `C` multiplies the data term (`C ~ 1/lambda`): **larger C = weaker regularization**.
On a *paraphrased* fraud message that matches no training template, C visibly trades
confidence against prior-collapse:

In [9]:
msg = ["There are trades on my account I never made, what do I do?"]
out = {}
for C in [10.0, 1.0, 0.1]:
    m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
                      LogisticRegression(max_iter=2000, C=C)).fit(texts, labels)
    out[f"C={C}"] = pd.Series(m.predict_proba(msg)[0], index=m.classes_).round(3)
print(pd.DataFrame(out))
print("\nC=0.1 under-fits toward the majority class -- and MISSES the fraud.")

                     C=10.0  C=1.0  C=0.1
account-access        0.039  0.121  0.226
fraud-report          0.695  0.429  0.172
general               0.239  0.355  0.406
transaction-dispute   0.027  0.095  0.196

C=0.1 under-fits toward the majority class -- and MISSES the fraud.


## 9. Calibration and the human-review band

The production design I recommend is **selective prediction**: auto-route confident tickets,
send low-confidence ones to a human. That only works if confidence has *dynamic range* — the
model must be measurably less sure when it is wrong. Out-of-fold check:

In [10]:
proba = cross_val_predict(baseline_config(), texts, labels, cv=gkf, groups=groups, method="predict_proba")
conf = proba.max(axis=1)
pred = np.array(sorted(set(labels)))[proba.argmax(axis=1)]
wrong = pred != labels

print(f"mean confidence {conf.mean():.3f} vs accuracy {(~wrong).mean():.4f} (mildly UNDER-confident out-of-fold)")
print(f"confidence on the {wrong.sum()} error(s): {np.round(conf[wrong], 3)}  |  median when right: {np.median(conf[~wrong]):.3f}")
for thr in [0.90, 0.80, 0.70]:
    band = conf < thr
    print(f"review band conf<{thr}: flags {band.mean()*100:5.1f}% of tickets, catches {int((band & wrong).sum())}/{int(wrong.sum())} errors")

mean confidence 0.921 vs accuracy 0.9975 (mildly UNDER-confident out-of-fold)
confidence on the 1 error(s): [0.362]  |  median when right: 0.941
review band conf<0.9: flags  22.2% of tickets, catches 1/1 errors
review band conf<0.8: flags   4.5% of tickets, catches 1/1 errors
review band conf<0.7: flags   1.5% of tickets, catches 1/1 errors


**Reading:** the one error arrives at confidence **0.362** against a median of ~0.94 when
correct — a wide, tunable gap. A `conf < 0.70` band already catches the $10k fraud miss while
flagging a modest share of traffic for humans. One nuance for later: `class_weight` distorts
probabilities (it deliberately trains on a re-weighted distribution), so if the review band is
the priority, the cleaner stack is *unweighted model + calibrated threshold chosen from the
cost matrix*. Either design beats silent auto-routing.

## 10. Alternatives — including Naive Bayes — measured, with significance

Naive Bayes is the natural challenger at n=400 (generative models converge faster on small
data). I benchmark it (both feature types), ComplementNB, LinearSVC and gradient boosting under
the *same* grouped CV, then ask the only question that matters: **can this dataset even detect
a difference?** McNemar's exact test on discordant pairs; the best achievable p-value with
b+c discordant pairs is 2 x 0.5^(b+c), so significance at 0.05 needs *at least 6 disagreements
all pointing one way*.

In [11]:
T = lambda **kw: TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
models = {
    "LogReg C=10 (shipped config)": baseline_config(),
    "LogReg C=10 + balanced":       baseline_config(class_weight="balanced"),
    "MultinomialNB on tf-idf":      make_pipeline(T(), MultinomialNB()),
    "MultinomialNB on counts":      make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB()),
    "ComplementNB on tf-idf":       make_pipeline(T(), ComplementNB()),
    "LinearSVC":                    make_pipeline(T(), LinearSVC()),
    "GradientBoosting":             make_pipeline(T(), GradientBoostingClassifier(random_state=SEED)),
}
preds, table = {}, []
for name, m in models.items():
    yp = cross_val_predict(m, texts, labels, cv=gkf, groups=groups)
    preds[name] = yp
    table.append({"model": name,
                  "acc": accuracy_score(labels, yp),
                  "macro_F1": f1_score(labels, yp, average="macro"),
                  "fraud_recall": recall_score(labels, yp, labels=["fraud-report"], average=None)[0],
                  "errors": int((yp != labels).sum())})
print(pd.DataFrame(table).round(4).to_string(index=False))

from scipy.stats import binomtest
base = preds["LogReg C=10 (shipped config)"]
print("\nMcNemar vs shipped config (b = alt fixes an error, c = alt breaks a correct one):")
for name, yp in preds.items():
    if name == "LogReg C=10 (shipped config)":
        continue
    b = int(((base != labels) & (yp == labels)).sum())
    c = int(((base == labels) & (yp != labels)).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    best = binomtest(b + c, b + c, 0.5).pvalue if b + c else 1.0
    print(f"  {name:<28} b={b} c={c:<3} exact p={p:.4f}  (best possible for this b+c: {best:.4f})")

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
n = NormalIndPower().solve_power(proportion_effectsize(0.9990, 0.9975), alpha=0.05, power=0.8, ratio=1, alternative="larger")
print(f"\nsamples per arm to *prove* 99.75% -> 99.90% at 80% power: {n:,.0f} (we have 400)")

nb_proba = cross_val_predict(models["MultinomialNB on counts"], texts, labels, cv=gkf, groups=groups, method="predict_proba")
print(f"NB confidence degeneracy: {(nb_proba.max(axis=1) > 0.99).mean()*100:.1f}% of predictions above 0.99 "
      f"(LogReg: {(conf > 0.99).mean()*100:.1f}%) -> no dynamic range for a review band")

                       model    acc  macro_F1  fraud_recall  errors
LogReg C=10 (shipped config) 0.9975    0.9961          0.98       1
      LogReg C=10 + balanced 1.0000    1.0000          1.00       0
     MultinomialNB on tf-idf 0.9700    0.9564          0.76      12
     MultinomialNB on counts 0.9975    0.9961          0.98       1
      ComplementNB on tf-idf 0.9975    0.9961          0.98       1
                   LinearSVC 0.9975    0.9961          0.98       1
            GradientBoosting 0.9925    0.9904          0.96       3

McNemar vs shipped config (b = alt fixes an error, c = alt breaks a correct one):
  LogReg C=10 + balanced       b=1 c=0   exact p=1.0000  (best possible for this b+c: 1.0000)
  MultinomialNB on tf-idf      b=0 c=11  exact p=0.0010  (best possible for this b+c: 0.0010)
  MultinomialNB on counts      b=0 c=0   exact p=1.0000  (best possible for this b+c: 1.0000)
  ComplementNB on tf-idf       b=0 c=0   exact p=1.0000  (best possible for this b+c: 1.000

**Reading:** the only statistically significant difference in the whole sweep is
**MultinomialNB-on-TF-IDF being *worse*** — and worse specifically on fraud (recall 0.76: TF-IDF's
fractional values break the multinomial count model, the likelihood flattens, and the 40%
`general` prior swallows the 12.5% class). NB on raw counts is byte-identical to the baseline;
so are ComplementNB and LinearSVC. Every "improvement" is unprovable *in principle*: the
baseline makes one error, so no alternative can ever reach 6 favorable disagreements here.
Model choice is not the lever. The evaluation instrument and the operating point are.

## 11. Hygiene candidate: `min_df=2` — tested, adopted

Dropping terms that appear in only one document removes every memorization hook from Section 6.
It cannot be justified by score (nothing can, per Section 10) — so it must be justified by
principle *and shown harmless*:

In [12]:
rows_out = []
for mdf in [1, 2]:
    for cw in [None, "balanced"]:
        m = baseline_config(min_df=mdf, class_weight=cw)
        yp = cross_val_predict(m, texts, labels, cv=gkf, groups=groups)
        nfeat = len(TfidfVectorizer(ngram_range=(1, 2), min_df=mdf, sublinear_tf=True).fit(texts).get_feature_names_out())
        rows_out.append({"min_df": mdf, "class_weight": str(cw), "features": nfeat,
                         "acc": accuracy_score(labels, yp),
                         "macro_F1": f1_score(labels, yp, average="macro"),
                         "fraud_recall": recall_score(labels, yp, labels=["fraud-report"], average=None)[0]})
print(pd.DataFrame(rows_out).round(4).to_string(index=False))
print("\n1,522 -> 1,135 features: all 387 single-document hooks gone, metrics identical. Adopted.")

 min_df class_weight  features    acc  macro_F1  fraud_recall
      1         None      1522 0.9975    0.9961          0.98
      1     balanced      1522 1.0000    1.0000          1.00
      2         None      1135 0.9975    0.9961          0.98
      2     balanced      1135 1.0000    1.0000          1.00

1,522 -> 1,135 features: all 387 single-document hooks gone, metrics identical. Adopted.


## 12. Embeddings — considered, rejected (for now)

Swapping TF-IDF for pretrained sentence embeddings is easy to wire and impossible to justify
here:

- **Undetectable benefit:** Section 10 shows no model change can reach significance on this
  dataset (the baseline's single error caps the discordant count at 1; six are needed), and
  proving even a 0.15 pp gain would take ~9,000 labeled samples.
- **Real costs:** a model download and heavier dependency for anyone running the repo, more
  latency per ticket, and a less inspectable failure mode than "which words voted for fraud".
- **The named trigger to revisit:** measured paraphrase/out-of-vocabulary misses on *real*
  traffic — the regime where lexical features genuinely fail. (Part B measures exactly such
  vocabulary drift in retrieval and handles it structurally instead of with embeddings; same
  ladder, same logic.)

## 13. Verdict

**I do not sign off on shipping this as-is** — not because 98.75% is inflated (the corrected
protocol scores 99.75–100%), but because the number is **unsupported as a ship signal**:

1. The evaluation protocol was invalid (leakage, single unstratified 80-row split, no error
   bars) — Wilson 95% CI [93.3%, 99.8%].
2. The benchmark is at ceiling: 400 templated, linearly separable synthetic messages. A perfect
   score on an easy benchmark is evidence about the benchmark.
3. The learned features are generator fingerprints, and hand-written benign messages already
   produce high-confidence false fraud reports (Section 7).
4. The report never looked at the class that matters: the one honest-protocol error is a
   **$10,000 fraud ticket routed to disputes**, and fraud recall at n=50 is known only to
   about ±7 pp.
5. `predict()` retrains the model on every call and deploys a different model than the one
   evaluated.

**The number I would defend to a PM:** fraud-report recall under template-grouped CV — 0.98
with the shipped config, 1.00 with class weights — with the caveat that the dataset cannot
certify either to better than ~±7 pp.

**The production metric:** a fraud-report recall floor (e.g. >= 0.95) at acceptable per-class
precision, monitored continuously — plus a low-confidence human-review band (Section 9), drift
monitoring on input text and confidence distributions, and agent re-routes as free labels.

**The minimal fix** (implemented in this repo, small-diff style): put the vectorizer inside a
`Pipeline` (leakage becomes impossible), `min_df=2` (hooks removed, measured harmless),
`class_weight='balanced'` (aligns the loss with the stated cost asymmetry), evaluate with
template-grouped + stratified CV and per-class metrics, and make `predict()` fit once and serve
many.

**The ship gate:** a shadow pilot on real traffic with the monitoring above — because nothing
measured on this synthetic set transfers by default.